# Universal Approximation Theorem
Universal approximation theorem serves as a foundation for using neural networks. Any continuous function can be approximated by a neural network with a  certain structure. We demonstrate the theorem by approximating some function using multi-layer perceptron.

My implementation is in Pytorch

###  Define the target function
We define a 2D statistical function that we want our neural network to learn:
$$
f(x_{1}, x_{2}) = \frac{(x_{1} - \bar{x})^{2} + {(x_{2} - \bar{x})^{2}}}{2}, \quad \bar{x} = \frac{x_{1} + x_{2}}{2}
$$
This functions computes the variance of two inputs

### Training Data

We randomly sample 1,000 points from the domain $[-2, 2] \times [-2, 2]$  and evaluate our target function at those sample points

### Neural Network

We define a two-layer neural network with 20 hidden units and ReLU activations. The network maps a two-dimensional input $(x_{1}, x_{2})$ to a scalar output, with trainable weights optimized during training.

### Model Training

We train the network using mean squared error (MSE) loss and the Adam optimizer with a learning rate of 0.001. Mini-batch stochastic gradient descent is used with a batch size of 100. Training is performed for 10,000 epochs.

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [0]:
# define the function to approximate
def F_mv(x1, x2):
    # return np.sin(3.14*x1/2.0) * np.cos(3.14 * x2/4.0)
    mean = (x1 + x2)/2
    return ((x1 - mean)**2 + (x2 - mean)**2)/2


# generate training data
A = 2
nb_samples = 1000

# Inputs
x_train = np.random.uniform(-A, A,  (nb_samples, 2)).astype(np.float32)
# outputs
y_train = F_mv(x_train[:, 0], x_train[:, 1]).reshape(-1, 1).astype(np.float32)

#convert to Tensors
X_train = torch.from_numpy(x_train)
Y_train = torch.from_numpy(y_train)

# define 2-layer neural network
class twolayernet(nn.Module):
  def __init__(self, input_dim = 2, hidden_dim = 20, output_dim = 1):
    super(twolayernet, self).__init__()
    self.hidden = nn.Linear(input_dim, hidden_dim)
    self.output = nn.Linear(hidden_dim, output_dim)
    self.ReLU = nn.ReLU()

  def forward(self, x):
    h = self.ReLU(self.hidden(x))
    y_pred = self.output(h)
    return y_pred

#create model
model = twolayernet()

#Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3)

# training loop
epochs = 10000
batch_size = 100

for epoch in range(epochs):
  #Mini batch
  idx = np.random.choice(nb_samples, batch_size, replace = False)
  inputs = X_train[idx]
  targets = Y_train[idx]

  #Forward pass
  outputs = model(inputs)
  loss = criterion(outputs, targets)

  # Backward pass and optimization
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()


#Visualize
def visualize_surface(model = None):
  grid_size = 100
  x = np.linspace(-A, A, grid_size)
  y = np.linspace(-A, A, grid_size)
  X, Y = np.meshgrid(x, y)

  Z_true = F_mv(X, Y)

  if model is not None:
        # Convert to torch tensor
        XY = np.stack([X.ravel(), Y.ravel()], axis=1).astype(np.float32)
        XY_torch = torch.from_numpy(XY)
        with torch.no_grad():
            Z_pred = model(XY_torch).numpy().reshape(grid_size, grid_size)
  else:
      Z_pred = None
  return X, Y, Z_true, Z_pred

X, Y, Z_true, Z_pred = visualize_surface(model)

# 2D heatmaps
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(Z_true, extent=(-A,A,-A,A), origin='lower')
plt.title("Original Function")
plt.colorbar()

plt.subplot(1,2,2)
plt.imshow(Z_pred, extent=(-A,A,-A,A), origin='lower')
plt.title("NN Approximation")
plt.colorbar()
plt.show()

# 3D wireframe plots
fig = plt.figure(figsize=(14,6))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_wireframe(X, Y, Z_true)
ax1.set_title("Original Function")

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_wireframe(X, Y, Z_pred)
ax2.set_title("NN Approximation")
plt.show()